# K-Ride Slim FastAPI — Kaggle + zrok

TorchServe/Celery 없이 모델 직접 로딩. zrok으로 퍼블릭 URL 노출.

**사전 준비:**
1. Kaggle Dataset `kride-data`에 업로드: `chroma_db/`, `models/ensemble_ranker.pkl`, `models/poi_cooccurrence_v2.pkl`
2. Kaggle Secrets에 환경변수 등록: `GROQ_API_KEY`, `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `SUPABASE_URL`, `SUPABASE_KEY`
3. 인터넷 접속 허용 (Settings → Internet → On)

## 1. 환경변수 설정 (Kaggle Secrets)

In [ ]:
import os

# Kaggle Secrets에서 환경변수 로드
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["GROQ_API_KEY"] = secrets.get_secret("GROQ_API_KEY")
    os.environ["NEO4J_URI"] = secrets.get_secret("NEO4J_URI")
    os.environ["NEO4J_USERNAME"] = secrets.get_secret("NEO4J_USERNAME")
    os.environ["NEO4J_PASSWORD"] = secrets.get_secret("NEO4J_PASSWORD")
    os.environ["SUPABASE_URL"] = secrets.get_secret("SUPABASE_URL")
    os.environ["SUPABASE_KEY"] = secrets.get_secret("SUPABASE_KEY")
    print("Kaggle Secrets 로드 완료")
except Exception as e:
    print(f"Kaggle Secrets 로드 실패 (로컬 .env 사용): {e}")

# 데이터 경로 설정
os.environ["CHROMA_PATH"] = "/kaggle/input/kride-data/chroma_db"
os.environ["MODELS_PATH"] = "/kaggle/input/kride-data/models"

print(f"GROQ_API_KEY: {'설정됨' if os.environ.get('GROQ_API_KEY') else '미설정'}")
print(f"NEO4J_URI: {'설정됨' if os.environ.get('NEO4J_URI') else '미설정'}")
print(f"SUPABASE_URL: {'설정됨' if os.environ.get('SUPABASE_URL') else '미설정'}")

## 2. 의존성 설치

In [ ]:
%%bash
pip install -q \
    fastapi uvicorn[standard] \
    sentence-transformers \
    chromadb \
    groq \
    neo4j \
    supabase \
    python-dotenv \
    numpy \
    pydantic

## 3. 데이터 확인

In [ ]:
import os

data_dir = "/kaggle/input/kride-data"
if os.path.exists(data_dir):
    for root, dirs, files in os.walk(data_dir):
        level = root.replace(data_dir, "").count(os.sep)
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 2:  # 2단계까지만 출력
            subindent = " " * 2 * (level + 1)
            for file in files[:10]:
                print(f"{subindent}{file}")
            if len(files) > 10:
                print(f"{subindent}... and {len(files) - 10} more files")
else:
    print(f"데이터 디렉토리 없음: {data_dir}")
    print("Kaggle Dataset 'kride-data'를 추가해주세요.")
    print("필요 파일: chroma_db/, models/ensemble_ranker.pkl, models/poi_cooccurrence_v2.pkl")

## 4. 서버 코드 복사

In [ ]:
# kaggle_server.py를 Kaggle Dataset에 포함했다면 복사
import shutil

server_src = "/kaggle/input/kride-data/kaggle_server.py"
server_dst = "/kaggle/working/kaggle_server.py"

if os.path.exists(server_src):
    shutil.copy2(server_src, server_dst)
    print(f"서버 코드 복사 완료: {server_dst}")
else:
    print(f"서버 코드 없음: {server_src}")
    print("직접 업로드하거나 아래 셀에서 수동 생성하세요.")

## 5. 모델 웜업 (선택사항)

서버 시작 전에 미리 모델을 로딩하면 첫 요청 응답이 빨라집니다.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")

from kaggle_server import get_embedder, get_reranker, get_chroma

# SentenceTransformer 로딩 (~30초)
embedder = get_embedder()
test_vec = embedder.encode(["서울 여행"], normalize_embeddings=True)
print(f"Embedder OK: dim={test_vec.shape}")

# CrossEncoder 로딩 (~20초)
reranker = get_reranker()
test_score = reranker.predict([["서울 맛집", "서울 강남 맛집 추천"]])
print(f"Reranker OK: score={test_score}")

# ChromaDB 확인
try:
    chroma = get_chroma()
    collections = chroma.list_collections()
    print(f"ChromaDB OK: {len(collections)} collections")
    for col in collections:
        print(f"  - {col.name}: {col.count()} docs")
except Exception as e:
    print(f"ChromaDB 연결 실패: {e}")

## 6. zrok 설치 + 터널 설정

In [ ]:
%%bash
# zrok 바이너리 다운로드
if [ ! -f /usr/local/bin/zrok ]; then
    echo "zrok 다운로드 중..."
    curl -sL -o /tmp/zrok.tar.gz https://github.com/openziti/zrok/releases/latest/download/zrok_linux_amd64.tar.gz
    tar -xzf /tmp/zrok.tar.gz -C /tmp/
    mv /tmp/zrok /usr/local/bin/zrok
    chmod +x /usr/local/bin/zrok
    echo "zrok 설치 완료"
else
    echo "zrok 이미 설치됨"
fi
zrok version

### 6-1. zrok 초대 토큰 설정

**최초 1회만 필요합니다.**
1. https://zrok.io 에서 무료 회원가입
2. 대시보드에서 Enable Token 복사
3. Kaggle Secrets에 `ZROK_TOKEN` 으로 등록

In [ ]:
import subprocess

# zrok enable (최초 1회)
zrok_token = os.environ.get("ZROK_TOKEN", "")
if not zrok_token:
    try:
        from kaggle_secrets import UserSecretsClient
        zrok_token = UserSecretsClient().get_secret("ZROK_TOKEN")
    except Exception:
        pass

if zrok_token:
    result = subprocess.run(
        ["zrok", "enable", zrok_token],
        capture_output=True, text=True
    )
    if result.returncode == 0 or "already enabled" in result.stderr:
        print("zrok enable 완료")
    else:
        print(f"zrok enable 결과: {result.stderr}")
else:
    print("ZROK_TOKEN이 설정되지 않았습니다.")
    print("Kaggle Secrets에 ZROK_TOKEN을 추가하세요.")
    print("https://zrok.io 에서 무료 가입 후 토큰 발급")

## 7. FastAPI 서버 시작 (백그라운드)

In [ ]:
import subprocess
import time

# FastAPI 백그라운드 실행
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "kaggle_server:app",
     "--host", "0.0.0.0", "--port", "8000", "--workers", "1"],
    cwd="/kaggle/working",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(5)  # 서버 시작 대기

# 서버 상태 확인
import urllib.request
try:
    with urllib.request.urlopen("http://localhost:8000/api/health") as resp:
        print(f"서버 상태: {resp.read().decode()}")
except Exception as e:
    print(f"서버 시작 실패: {e}")
    print("stderr:", server_proc.stderr.read().decode()[:500])

## 8. zrok 터널 시작

이 셀을 실행하면 퍼블릭 URL이 출력됩니다. 이 URL로 외부에서 접속 가능합니다.

In [ ]:
import subprocess
import time
import re

# zrok share (퍼블릭 URL 생성)
zrok_proc = subprocess.Popen(
    ["zrok", "share", "public", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# URL 추출 대기
public_url = None
for _ in range(30):  # 최대 30초 대기
    line = zrok_proc.stdout.readline()
    if line:
        print(line.strip())
        match = re.search(r'https://[\w-]+\.share\.zrok\.io', line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(1)

if public_url:
    print(f"\n{'='*60}")
    print(f"K-Ride API 퍼블릭 URL: {public_url}")
    print(f"{'='*60}")
    print(f"\n테스트 명령어:")
    print(f"  curl {public_url}/api/health")
    print(f"  curl {public_url}/api/artists")
    print(f"  curl -X POST {public_url}/api/recommend/ai \\")
    print(f"    -H 'Content-Type: application/json' \\")
    print(f"    -d '{{\"artists\":[\"BTS\"], \"regions\":[\"서울\"], \"purposes\":[\"kculture\"]}}'")
else:
    print("zrok URL 추출 실패. 로그를 확인하세요.")

## 9. API 테스트

In [ ]:
import requests
import json

BASE = "http://localhost:8000"  # 로컬 테스트
# BASE = public_url  # zrok URL로 외부 테스트

# 1. Health
print("=== /api/health ===")
r = requests.get(f"{BASE}/api/health")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

# 2. Artists
print("\n=== /api/artists ===")
r = requests.get(f"{BASE}/api/artists")
artists = r.json()["artists"]
print(f"{len(artists)}개 아티스트")
for a in artists[:5]:
    print(f"  - {a.get('name', a.get('id', '?'))}")

# 3. Regions
print("\n=== /api/regions ===")
r = requests.get(f"{BASE}/api/regions")
regions = r.json()["regions"]
print(f"{len(regions)}개 지역")
for rg in regions[:5]:
    print(f"  - {rg['name']}")

In [ ]:
# 4. AI 추천
print("=== /api/recommend/ai ===")
r = requests.post(f"{BASE}/api/recommend/ai", json={
    "artists": ["BTS"],
    "regions": ["서울"],
    "purposes": ["kculture"],
})
data = r.json()
print(f"POI {data['count']}개")
for p in data["pois"][:3]:
    print(f"  - {p.get('name', '?')} ({p.get('sido', '?')})")
print(f"\n추천 텍스트:\n{data.get('recommendation_text', '')[:300]}")

In [ ]:
# 5. 일정 생성
print("=== /api/recommend/itinerary ===")
r = requests.post(f"{BASE}/api/recommend/itinerary", json={
    "duration": "1박2일",
    "artists": ["BTS"],
    "regions": ["서울"],
    "purposes": ["kculture", "food"],
})
data = r.json()
print(json.dumps(data, indent=2, ensure_ascii=False)[:1000])

In [ ]:
# 6. 챗봇
print("=== /api/chatbot ===")
r = requests.post(f"{BASE}/api/chatbot", json={
    "message": "서울에서 BTS 관련 여행지 추천해줘",
    "session_id": "test1",
})
data = r.json()
print(f"Reply:\n{data['reply'][:500]}")
print(f"\nSources: {data['sources']}")
print(f"POIs: {len(data['pois'])}개")

## 10. 서버 종료

테스트가 끝나면 서버와 zrok을 종료하세요.

In [ ]:
# 서버 & zrok 종료
try:
    zrok_proc.terminate()
    print("zrok 터널 종료")
except Exception:
    pass

try:
    server_proc.terminate()
    print("FastAPI 서버 종료")
except Exception:
    pass